# Stateful operators

Stateful operators bring the real fun to Kafi Streams. The most interesting operators are of course `join_equi()` and `join()` (equi join and general/non-equi join) and `group_by_agg()` (group by + aggregate). `agg()` is just a special case of `group_by_agg()` (obviously, without grouping).

Kafi Streams also already supports the stateful set operators `distinct()`, `union()`, `intersect()` and `minus()`.

## Overview

  * [join_equi()](#join_equi-operator)
  * [join()](#join-operator)
  * [group_by_agg()](#group_by_agg-operator)
    * [group_by_sum()](#group_by_sum-operator)
    * [group_by_max()](#group_by_max-operator)
    * [group_by_min()](#group_by_min-operator)
    * [group_by_avg()](#group_by_avg-operator)
    * [group_by_count()](#group_by_count-operator)
    * [agg()](#agg-operator)
      * [sum()](#sum-operator)
      * [max()](#max-operator)
      * [min()](#min-operator)
      * [avg()](#avg-operator)
      * [count()](#count-operator)
  * [distinct()](#distinct-operator)
  * [union()](#union-operator)
  * [intersect()](#intersect-operator)
  * [minus()](#minus-operator)


## Preparation

Before we start off, we first prepare for the examples to follow:

In [ ]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, "../")
sys.path.insert(1, "../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator, CustomerGenerator
click_generator = ClickGenerator()
customer_generator = CustomerGenerator()

click_source_str = "clicks"
customer_source_str = "customers"


Please also note that when we re-use the same example over and over again to illustrate how the operators work, we always mark the important new parts as follows:
```python
    # <------------------------------>
    ...important new parts...
    # <------------------------------>
```

<a id="join_equi-operator"></a>
## join_equi()

Probably the most popular stateful operator - `join_equi()` is the equi join operator of Kafi Streams:
```
join_equi(right_tn, left_key_fun, right_key_fun, project_fun, **kwargs)
```
* `right_tn`: the right side of the join.
* `left_key_fun: l_r -> any`: the selection function getting an input record an returning the left join key
* `right_key_fun: r_r -> any`: the selection function for the right join key
* `project_fun: l_r, r_r -> r`: the projection function; gets both the left and right input records and returns the output (=projection) record of the join.

Note that the `left_key_fun` and `right_key_fun` can in principle select *anything* from the incoming records. As Kafi Streams is not tied to Kafka but just receives a Python dictionary, there are no restriction to Kafka keys or so as e.g. in Kafka Streams.

Here is an example:

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .join_equi(customer_tn,
               left_key_fun=lambda l_r: l_r["customer_id"],
               right_key_fun=lambda r_r: r_r["id"],
               project_fun=lambda l_r, r_r: {"customer_id": l_r["customer_id"],
                                             "view_time": l_r["view_time"],
                                             "name": r_r["name"]})
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In this example topology, we join the clicks and customers as before in the [Quickstart](../quickstart.ipynb).

The example data consists of three clicks and two customers. Two of the clicks match a customer from the right side (`customer_id` = `42`) and thus we receive both clicks (joined with the `name`) in the output.

<a id="join-operator"></a>
## join()

While `join_equi()` might be the most popular stateful operator, Kafi Streams also offers a general/non-equi join with the `join()` operator. Whereas with `join_equi()`, you can only select a key from each side of the join, `join()` allows you to specify any predicate:
```
join(right_tn, predicate_fun, project_fun, **kwargs)
```
* `right_tn`: the right side of the join
* `predicate_fun: l_r, r_r -> bool`: the join predicate - a function getting both the left and right input records and returning a bool.
* `project_fun: l_r, r_r -> r`: the projection function; gets both the left and right input records and returns the output (=projection) of the join.

As in the database world as well, non-equi join operator (`join()`) is, on the one hand, more flexible than `join_equi()`, but on the other hand much less performant:
* `join_equi` is based on hash maps (complexity `O(1)` per row)
* `join` cannot use hash maps (`O(N)` per row)

So whenever you can, use `join_equi()`, and `join()` only if you need the extra flexibility or the predicate can be calculated quickly enough (e.g. in case one of the two sides of the join always has only few elements).

An example is in order for the `join()` operator as well:

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"], "name": r["value"]["name"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .join(customer_tn,
          predicate_fun=lambda l_r, r_r: l_r["customer_id"] == r_r["id"] and l_r["view_time"] > 60,
          project_fun=lambda l_r, r_r: {"customer_id": l_r["customer_id"],
                                        "view_time": l_r["view_time"],
                                        "name": r_r["name"]})
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Except for the change from `join_equi()` to `join`, the topology is the same as in the previous example for `join_equi()`. The join predicate also works a little differently to the two selections in the `join_equi()` example: This time, we do not only match the customer IDs but also add the condition that the `view_time` of the click must be greater than `60`.

The example data is the same as in the previous example for `join_equi()`. But the output is different, because even though the second click of customer `42` does match the customer ID of one of the customers, its `view_time` is not greater than `60`. Thus we only receive one output record.

<a id="group_by_agg-operator"></a>
## group_by_agg()

The `group_by_agg()` operator is, as its name implies, a combination of grouping by some key and an arbitrary aggregation:
```
group_by_agg(key_fun, value_fun, agg_fun, agg_initial, project_fun, **kwargs)
```
* `key_fun: r -> any`: the selection function for the key for the grouping
* `value_fun: r -> any`: the selection function for the value of the aggregation
* `agg_fun: agg_any, value_any -> any`: the aggregation function; gets the result of the aggregation so far (`agg_any`) and the next selected value to aggregate (`value_any`), and returns the updated aggregation.
* `agg_initial_any`: the initial value of the aggregation
* `project_fun: key_any, agg_any -> r`: the projection function. Gets both the selected key and the aggregation result for that key and returns the output (=projection) of the group by + aggregation.

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .group_by_agg(key_fun=lambda r: r["customer_id"],
                  value_fun=lambda r: r["view_time"],
                  agg_fun=lambda agg_any, value_any: {"view_times": agg_any["view_times"] + [value_any],
                                                      "sum_view_times": agg_any["sum_view_times"] + value_any},
                  agg_initial_any={"view_times": [],
                                   "sum_view_times": 0},
                  project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                        "view_times": agg_any["view_times"],
                                                        "sum_view_times": agg_any["sum_view_times"]})
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In the example topology, we group by `customer_id` and aggregate the `view_time` of our clickstream input.

The actual aggregation function does two things:
1. It collects each `view_time` per customers in the field `view_times`.
2. It sums up each `view_time` per customer in the field `sum_view_times`.

The projection function returns records with three fields:
1. `customer_id`: the group/key
2. `view_times`: the collected view times
3. `sum_view_times`: the sum of the view times

In the example data, customer `42` has two clicks: one with a `view_time` of `67` and one with `23`. Hence, after the aggregation, the `view_times` are `[67, 23]` and the sum is `90`.


<a id="group_by_sum-operator"></a>
### group_by_sum()

`group_by_sum()` is syntactic sugar for `group_by_agg()` where the aggregation function sums up the selected values:
```
group_by_sum(key_fun, value_fun, project_fun, sum_initial_any=0, **kwargs)
```
* `key_fun: r -> any`: the selection function for the key for the grouping
* `value_fun: r -> any`: the selection function for the value of the aggregation
* `project_fun: key_any, agg_any -> r`: the projection function. Gets both the selected key and the aggregation result for that key and returns the output (=projection) of the group by + aggregation.
* `sum_initial_any`: the initial value of the aggregation (default: `0`)

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .group_by_sum(key_fun=lambda r: r["customer_id"],
                  value_fun=lambda r: r["view_time"],
                  project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                        "sum_view_times": agg_any})
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="group_by_max-operator"></a>
### group_by_max()

`group_by_max()` is syntactic sugar for `group_by_agg()` where the aggregation function calculates the maximum of the selected values:
```
group_by_max(key_fun, value_fun, project_fun, max_initial_any=0, **kwargs)
```
* `key_fun: r -> any`: the selection function for the key for the grouping
* `value_fun: r -> any`: the selection function for the value of the aggregation
* `project_fun: key_any, agg_any -> r`: the projection function. Gets both the selected key and the aggregation result for that key and returns the output (=projection) of the group by + aggregation.
* `max_initial_any`: the initial value of the aggregation (default: `0`)

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .group_by_max(key_fun=lambda r: r["customer_id"],
                  value_fun=lambda r: r["view_time"],
                  project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                        "max_view_times": agg_any})
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="group_by_min-operator"></a>
### group_by_min()

`group_by_min()` is syntactic sugar for `group_by_agg()` where the aggregation function calculates the minimum of the selected values:
```
group_by_max(key_fun, value_fun, project_fun, min_initial_any=sys.maxsize, **kwargs)
```
* `key_fun: r -> any`: the selection function for the key for the grouping
* `value_fun: r -> any`: the selection function for the value of the aggregation
* `project_fun: key_any, agg_any -> r`: the projection function. Gets both the selected key and the aggregation result for that key and returns the output (=projection) of the group by + aggregation.
* `min_initial_any`: the initial value of the aggregation (default: `sys.maxsize` = the highest possible integer)

Here is an example:

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .group_by_min(key_fun=lambda r: r["customer_id"],
                  value_fun=lambda r: r["view_time"],
                  project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                        "min_view_times": agg_any})
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="group_by_avg-operator"></a>
### group_by_avg()

`group_by_avg()` is syntactic sugar for `group_by_agg()` where the aggregation function calculates the average of the selected values:
```
group_by_avg(key_fun, value_fun, project_fun, **kwargs)
```
* `key_fun: r -> any`: the selection function for the key for the grouping
* `value_fun: r -> any`: the selection function for the value of the aggregation
* `project_fun: key_any, agg_any -> r`: the projection function. Gets both the selected key and the aggregation result for that key and returns the output (=projection) of the group by + aggregation.

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .group_by_avg(key_fun=lambda r: r["customer_id"],
                  value_fun=lambda r: r["view_time"],
                  project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                        "avg_view_times": agg_any})
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="group_by_count-operator"></a>
### group_by_count()

`group_by_count()` is syntactic sugar for `group_by_agg()` where the aggregation function counts the grouped values.
```
group_by_count(key_fun, project_fun, **kwargs)
```
* `key_fun: r -> any`: the selection function for the key for the grouping
* `project_fun: key_any, agg_any -> r`: the projection function. Gets both the selected key and the aggregation result for that key and returns the output (=projection) of the group by + aggregation.

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .group_by_count(key_fun=lambda r: r["customer_id"],
                    project_fun=lambda key_any, agg_any: {"customer_id": key_any,
                                                          "count_view_times": agg_any})
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="agg-operator"></a>
### agg()

`agg()` is syntactic sugar for `group_by_agg()` which just aggregates (there is only one group/key),
```
agg(value_fun, agg_fun, agg_initial_any, project_fun, **kwargs)
```
* `value_fun: r -> any`: the selection function for the value of the aggregation
* `agg_fun: agg_any, value_any -> any`: the aggregation function; gets the result of the aggregation so far (`agg_any`) and the next selected value to aggregate (`value_any`), and returns the updated aggregation.
* `agg_initial_any`: the initial value of the aggregation
* `project_fun: agg_any -> r`: the projection function. Gets the aggregation result and returns the output (=projection) of the aggregation.
* `min_initial_any`: the initial value of the aggregation (default: `sys.maxsize` = the highest possible integer)

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .agg(value_fun=lambda r: r["view_time"],
         agg_fun=lambda agg_any, value_any: {"view_times": agg_any["view_times"] + [value_any],
                                             "sum_view_times": agg_any["sum_view_times"] + value_any},
         agg_initial_any={"view_times": [],
                          "sum_view_times": 0},
         project_fun=lambda agg_any: {"view_times": agg_any["view_times"],
                                      "sum_view_times": agg_any["sum_view_times"]})
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="sum-operator"></a>
### sum()

`sum()` is syntactic sugar for `agg()` where the aggregation function sums up the selected values:
```
sum(value_fun, project_fun=lambda agg_any: agg_any, sum_initial_any=0, **kwargs)
```
* `value_fun: r -> any`: the selection function for the value of the aggregation
* `project_fun: agg_any -> r`: the projection function. Gets the aggregation result and returns the output (=projection) of the aggregation. Default: `lambda agg_any: agg_any` (identity function).
* `sum_initial_any`: the initial value of the aggregation (default: `0`)

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------
    .sum(value_fun=lambda r: r["view_time"])
    # ------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="max-operator"></a>
### max()

`max()` is syntactic sugar for `agg()` where the aggregation function calculates the maximum of the selected values:
```
max(value_fun, project_fun=lambda agg_any: agg_any, max_initial_any=0, **kwargs)
```
* `value_fun: r -> any`: the selection function for the value of the aggregation
* `project_fun: agg_any -> r`: the projection function. Gets the aggregation result and returns the output (=projection) of the aggregation. Default: `lambda agg_any: agg_any` (identity function).
* `max_initial_any`: the initial value of the aggregation (default: `0`)

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------
    .max(value_fun=lambda r: r["view_time"])
    # ------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="min-operator"></a>
### min()

`min()` is syntactic sugar for `agg()` where the aggregation function calculates the minimum of the selected values:
```
min(value_fun, project_fun=lambda agg_any: agg_any, min_initial_any=0, **kwargs)
```
* `value_fun: r -> any`: the selection function for the value of the aggregation
* `project_fun: agg_any -> r`: the projection function. Gets the aggregation result and returns the output (=projection) of the aggregation. Default: `lambda agg_any: agg_any` (identity function).
* `min_initial_any`: the initial value of the aggregation (default: `0`)

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------
    .min(value_fun=lambda r: r["view_time"])
    # ------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="avg-operator"></a>
### avg()

`avg()` is syntactic sugar for `agg()` where the aggregation function calculates the average of the selected values:
```
avg(value_fun, project_fun=lambda agg_any: agg_any, **kwargs)
```
* `value_fun: r -> any`: the selection function for the value of the aggregation
* `project_fun: agg_any -> r`: the projection function. Gets the aggregation result and returns the output (=projection) of the aggregation. Default: `lambda agg_any: agg_any` (identity function).

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------
    .avg(value_fun=lambda r: r["view_time"])
    # ------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="count-operator"></a>
### count()

`count()` is syntactic sugar for `agg()` where the aggregation function counts the the selected values:
```
count(value_fun, project_fun=lambda agg_any: agg_any, **kwargs)
```
* `project_fun: agg_any -> r`: the projection function. Gets the aggregation result and returns the output (=projection) of the aggregation. Default: `lambda agg_any: agg_any` (identity function).

Here is an example:


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"], "view_time": r["value"]["view_time"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------
    .count()
    # ------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


<a id="distinct-operator"></a>
## distinct()

The `distinct()` operator removes duplicate records, ensuring each unique record is represented only once. It has no counterpart in Kafka Streams.

Now you might ask why we would need such an operator in a relational stream processing engine working on sets anyway. Or why this operator is categorized as stateful and not stateless.

In DBSP/pydbsp, we actually work on streams of *deltas* of ZSets. A delta is a batch of records of any size.

If we just consider one delta, i.e., one batch of records, the relational, set-based nature of DBSP of course avoids duplicates natively. But if we receive another batch of records, DBSP can only detect whether it has already seen the individual records if it stores the previous delta/batch of records in its state.

To the syntax:
```
distinct(**kwargs)
```

And to an example.

In [ ]:
# ------------------------------>

click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"customer_id": r["value"]["customer_id"]})
)


non_distinct_sink_str = "non_distinct"
non_distinct_tn = click_tn.sink(non_distinct_sink_str)

distinct_sink_str = "distinct"
distinct_tn = (
    click_tn
    .distinct()
    .sink(distinct_sink_str)
)

built_tn = Tn.build(non_distinct_tn, distinct_tn)

# ------------------------------>

print("Step 1")

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 4711, 'view_time': 76, 'ts': 1786618968910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

sink_str_output_m_list_dict = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for sink_str, output_m_list in sink_str_output_m_list_dict.items():
    print(sink_str, output_m_list)

#

print("\nStep 2")

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

sink_str_output_m_list_dict = built_tn.process({click_source_str: click_input_m_list})
print("\nOutput:")
for sink_str, output_m_list in sink_str_output_m_list_dict.items():
    print(sink_str, output_m_list)


In the example, we first select only the `customer_id` of the inputs.

Then create two sinks:
* `non_distinct`: Here we do not use the `distinct()` operator.
* `distinct`: Here we do use it.

Now we feed in three messages with the same customer ID (=a duplicate output) in two steps:
1. We feed the first two messages. In the output you can see that both the `non_distinct` and the `distinct` sink are both perfectly de-duplicated.
2. We feed the third message. Now the difference becomes visible:
  * `non_distinct`: Since this sink does *not* use the `distinct()` operator, it has forgotten that the third message is actually a duplicate and returns it.
  * `distinct`: For this sink, we *do* use the `distinct()` operator. And you can see that it has correctly detected the duplicate and hence does not return any output.


<a id="union-operator"></a>
## union()

The `union` operator takes two input streams and returns their union as in set theory:
```
union(other_tn, **kwargs)
```
* `other_tn`: the other side of the union

Once again, I think an example is the best explanation.

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    # <------------------------------>
    .map(lambda r: {"id": r["value"]["customer_id"]})
    # <------------------------------>
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    # <------------------------------>
    .map(lambda r: {"id": r["value"]["id"]})
    # <------------------------------>
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .union(customer_tn)
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


In the topology, we first select only the `customer_id` of the inputs and return it with the same field name `id`.

Then we use the `union()` operator.

The two input sets are:
* from clicks: `{{'id': 42'}, {'id': 24}}`
* from customers: `{{'id': 42'}, {'id': 4711}}`

The union of these two sets is, as in the output from Kafi Streams:
```
{
    {'id': 42'}, 
    {'id': 24},
    {'id': 4711}
}
```


<a id="intersect-operator"></a>
## intersect()

The `intersect` operator takes two input streams and returns their intersection as in set theory:
```
intersect(other_tn, **kwargs)
```
* `other_tn`: the other side of the intersection

An example is coming up...

In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"id": r["value"]["customer_id"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .intersect(customer_tn)
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Here, we utilize the `intersect()` operator.

The two input sets are (again):
* from clicks: `{{'id': 42'}, {'id': 24}}`
* from customers: `{{'id': 42'}, {'id': 4711}}`

The intersection of these two sets is, as in the output from Kafi Streams:
```
{
    {'id': 42'}, 
}
```


<a id="minus-operator"></a>
## minus()

The `minus` operator takes two input streams and returns their difference as in set theory.
```
minus(right_tn, **kwargs)
```
* `right_tn`: the right side of the set difference (the subtrahend)

To an example.


In [ ]:
click_tn = (
    Tn.source(click_source_str)
    #
    .map(lambda r: {"id": r["value"]["customer_id"]})
)

customer_tn = (
    Tn.source(customer_source_str)
    #
    .map(lambda r: {"id": r["value"]["id"]})
)

built_tn = Tn.build(
    click_tn
    # <------------------------------>
    .minus(customer_tn)
    # <------------------------------>
)

click_input_m_list = [{'key': None, 'value': {'customer_id': 42, 'view_time': 67, 'ts': 1786618958910}},
                      {'key': None, 'value': {'customer_id': 24, 'view_time': 76, 'ts': 1786618968910}},
                      {'key': None, 'value': {'customer_id': 42, 'view_time': 23, 'ts': 1786618978910}}]
print("Input (clicks):")
for m in click_input_m_list:
    print(m)

customer_input_m_list = [{'key': '42', 'value': {'id': 42, 'name': 'Betty Graham MD'}},
                         {'key': '4711', 'value': {'id': 4711, 'name': 'Frank Frank'}}]
print("\nInput (customers):")
for m in customer_input_m_list:
    print(m)

output_m_list = built_tn.process({click_source_str: click_input_m_list, customer_source_str: customer_input_m_list})
print("\nOutput:")
for m in output_m_list:
    print(m)


Here we use the `minus()` operator.

The two input sets are (again):
* from clicks: `{{'id': 42'}, {'id': 24}}`
* from customers: `{{'id': 42'}, {'id': 4711}}`

The set difference of these sets from clicks and customeers is, as in the output from Kafi Streams:
```
{
    {'id': 24'}
}
```
